# SmartHausGroup.github Repo CI/CD Conformance

Purpose: prove `SmartHausGroup.github` follows the SMARTHAUS Repo CI/CD Conformance Standard.

Lemma mapping: L-SmartHausGroup.github-CICD-1 through L-SmartHausGroup.github-CICD-5.

Invariant support: `invariants/cicd_conformance.yaml`.

Expected result: deterministic pass with seed `1729`.


## Setup

Why: bind the proof to the repository root and deterministic seed.

Alternatives rejected: implicit random state and remote-only checks.

Invariant enforced: fail-closed deterministic evaluation.

Correctness: the repo root and seed are stable.


In [ ]:
from pathlib import Path
import json

SEED = 1729
ROOT = Path.cwd()
if ROOT.name == "ma":
    ROOT = ROOT.parents[1]
elif not (ROOT / "configs/ucp/repo_cicd.yaml").exists():
    ROOT = Path("/Users/smarthaus/Projects/GitHub/SmartHausGroup.github")

assert SEED == 1729
assert (ROOT / "configs/ucp/repo_cicd.yaml").exists()


## Manifest Binding

Why: the repo must declare its profile, checks, Make targets, evidence, locks, and UCP receipt policy.

Alternatives rejected: workflow-only declarations without a UCP-readable manifest.

Invariant enforced: manifest binding.

Correctness: required manifest keys and targets are present.


In [ ]:
manifest_text = (ROOT / "configs/ucp/repo_cicd.yaml").read_text()
for token in ["name: SmartHausGroup.github", "repo-cicd-conformance", "evidence-pack", "ai-validate-full", "prepare-publication", "push-safe", "requires_ucp_receipt: true"]:
    assert token in manifest_text, token
for path in ["configs/ucp/standard_sources.lock", "configs/ucp/standard_toolchain.lock", "artifacts/scorecards/cicd_conformance.json"]:
    assert (ROOT / path).exists(), path


## Branch Non-Bypass

Why: protected branches must reject direct mutation.

Alternatives rejected: local hooks only, because hooks can be bypassed.

Invariant enforced: branch non-bypass.

Correctness: the ruleset requires PRs, signed commits, linear history, required checks, and no bypass actors.


In [ ]:
ruleset = json.loads((ROOT / ".github/rulesets/protected-branches.json").read_text())
rule_types = {rule["type"] for rule in ruleset["rules"]}
for required in ["pull_request", "non_fast_forward", "deletion", "required_linear_history", "required_signatures", "required_status_checks"]:
    assert required in rule_types, required
assert ruleset["bypass_actors"] == []
checks = ruleset["rules"][-1]["parameters"]["required_status_checks"]
assert {check["context"] for check in checks} == {"repo-cicd-conformance", "evidence-pack"}


## Fail-Closed Gates

Why: required workflow jobs must run deterministic local targets and fail on missing evidence.

Alternatives rejected: advisory-only gates and warning-only status checks.

Invariant enforced: fail-closed required gates.

Correctness: workflow and Make targets invoke the verifier and pack verification.


In [ ]:
workflow = (ROOT / ".github/workflows/repo-cicd-conformance.yml").read_text()
for token in ["repo-cicd-conformance", "evidence-pack", "make repo-cicd-conformance", "make ai-pack-verify"]:
    assert token in workflow, token
make_ai = (ROOT / "Makefile.ai").read_text()
for token in ["repo-cicd-conformance:", "ai-evidence-pack:", "ai-pack-verify:", "prepare-publication:", "push-safe:"]:
    assert token in make_ai, token


## Evidence Traceability

Why: adoption must bind docs, invariants, notebook, scorecard, and UCP receipt policy.

Alternatives rejected: green workflow without traceable proof artifacts.

Invariant enforced: evidence traceability and UCP receipt binding.

Correctness: all evidence files exist and the scorecard is green and seed locked.


In [ ]:
evidence = [
    "docs/ma/cicd_conformance_phase0_intent.md",
    "docs/ma/cicd_conformance_phase1_formula.md",
    "docs/ma/cicd_conformance_phase2_calculus.md",
    "docs/ma/cicd_conformance_phase3_lemmas.md",
    "invariants/cicd_conformance.yaml",
    "notebooks/ma/cicd_conformance.ipynb",
    "artifacts/scorecards/cicd_conformance.json",
]
missing = [path for path in evidence if not (ROOT / path).exists()]
assert missing == [], missing
scorecard = json.loads((ROOT / "artifacts/scorecards/cicd_conformance.json").read_text())
assert scorecard["status"] == "GREEN"
assert scorecard["determinism"]["seed_locked"] is True
assert scorecard["score"]["passed"] == scorecard["score"]["total"]
